In [2]:
import pandas as pd
import numpy as np
from functools import reduce

In [55]:
# DATA IMPORTS
# 4/20/2026
# Available data - 
# Pumps - psi, runtime, disposal volume, month of year
# Electrical - date, billed kwh, total due/final bill
# Well workovers - date, well name, description of work done
# Maintenance data - 

# Notes - 
# Billed DM == peak kilowatts for time period of site
# 8.48% kwh rate increase in June 2022

# Ideas
# 1 - explore psi/kwh relationship and psi/billing relationship for each well and aggregated wells
# 2 - Calculate cost per kwh for each month for each well
# 3 - For predictions/explanations - duplicate and normalize data columns, so that
#     one data column is used for display and another for prediction/explanation.
# 4 - get a count of workovers by well and compare that count to run hours and disposal volume
#     - which wells have the highest disposal volume per amount of repairs
# 5 - quantify well workover cost to compare repair cost to disposal income


# Target metric - Cost per barrels disposed for each well****
#               - kwh/barrel for each well 
#               - $/kwh for each well and field wide
#               - kwh/barrel * $/kwh = $/barrel
# 

# electrical billing data
elec_df = pd.read_csv(r"C:\Users\jroland\East Texas Salt Water Disposal Company\Operations - Documents\Engineering\Disposal Wells\Workover Analysis\Electrical Billing Data Processed Foundation.csv")

# drop comments
exclude = ("Comments", "Total Due")
include = [col for col in elec_df.columns if col not in exclude]

elec_df = elec_df[include]


# pump hours and disposal volume data
pumps_df = pd.read_excel(r"C:\Users\jroland\East Texas Salt Water Disposal Company\Operations - Documents\Engineering\Disposal Wells\Workover Analysis\Pump Summary Back End Data.xlsx",
                         sheet_name="Pump Summary Back End Data")

# well work data
well_work_df = pd.read_excel(r"C:\Users\jroland\East Texas Salt Water Disposal Company\Operations - Documents\Engineering\Disposal Wells\Workover Analysis\Pump Summary Back End Data.xlsx",
                             sheet_name="Well Work History")

# well install dates with pump numbers
well_install_df = pd.read_excel(r"C:\Users\jroland\East Texas Salt Water Disposal Company\Operations - Documents\Engineering\Disposal Wells\Workover Analysis\Pump Summary Back End Data.xlsx",
                           sheet_name="Wells")

# well and pump maintenance costs
maintenance_df = pd.read_excel(r"C:\Users\jroland\East Texas Salt Water Disposal Company\Operations - Documents\Engineering\Disposal Wells\Workover Analysis\Pump and Well Maintenance.xlsx",
                               sheet_name="PD + REDA Combined")

# monthly revenue data
revenue_df = pd.read_excel(r"C:\Users\jroland\East Texas Salt Water Disposal Company\Operations - Documents\Engineering\Disposal Wells\Workover Analysis\ETSWD Monthly Revenue.xlsx",
                          sheet_name="Monthly Totals")

# We're going to need to match the names between electrical billing and pumps - take note of which sites
# have aggregated billing Blum/Blum6/Blum9 for example



In [56]:
# TEXT PRE-PROCESSING

# Figure out best scheme for joining data 
# First - match up well run hours and water disposal with electrical billing
#       - drop instances of billing that are not in the disposal volumes data

# check NAs
print(elec_df['Service Address'].isna().sum())

# drop NAs for the well address - pretty sure this is from deleting the Silvey Booster from records
elec_df = elec_df.dropna(subset=['Service Address'])
print(elec_df['Service Address'].isna().sum())

# Fix the spacing between # and the well number in the electrical data
elec_names_filter = elec_df['Service Address'].str.contains(r"\d", regex=True)
print(elec_names_filter)
filt_elec_df = elec_df[elec_names_filter]

# The space between the # sign and number always occurs in the same place with respect
# to the end of the name
# assign the digit to a match group and replace it with the group number.
# r"\d" does not work in the replacement string because the match and replacement strings support different syntax
# \d, \b, \w, [], $, etc are pattern matching operations. We don't need to pattern match in the replacement, we 
# need to substitute!
filt_elec_df['Service Address'] = filt_elec_df['Service Address'].str.replace(r" (\d)", r"\g<1>", regex=True)

# fix entries for well names in pumps_df that start with ZZZ
pumps_df['Well'] = pumps_df['Well'].str.replace(r"ZZZ", r"", regex=True)

# Create a sequence of unique well names from pumps_df
well_names_filter = pumps_df['Well'].unique()

# Align the column names for wells among dataframes to allow for merging later.
# Matching field for each df
# elec_df -> Service Address, pumps_df -> Well, well_work_df -> Well, well_install_df -> Well Name, maintenance_df -> Location
pumps_df['Name'] = pumps_df['Well']
well_work_df['Name'] = well_work_df['Well']
filt_elec_df['Name'] = filt_elec_df['Service Address']
well_install_df['Name'] = well_install_df['Well Name']
maintenance_df['Name'] = maintenance_df['Location']

# Remove any trailing whitespaces in any of the dataframes.
df_list = [filt_elec_df, pumps_df, well_work_df, well_install_df, maintenance_df]
name_list = ["elec_df", "pump_df", "well_work_df", "well_install_df", "maintenance_df"]

for df, name in zip(df_list, name_list):
    df["Name"] = df["Name"].str.strip()
    df["Name"] = df["Name"].str.upper()
    

89
0
0         True
1         True
2         True
3         True
4         True
         ...  
18623    False
18624     True
18625     True
18626    False
18627     True
Name: Service Address, Length: 18539, dtype: bool


C:\Users\jroland\AppData\Local\Temp\ipykernel_29044\3882918921.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filt_elec_df['Service Address'] = filt_elec_df['Service Address'].str.replace(r" (\d)", r"\g<1>", regex=True)
C:\Users\jroland\AppData\Local\Temp\ipykernel_29044\3882918921.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filt_elec_df['Name'] = filt_elec_df['Service Address']
C:\Users\jroland\AppData\Local\Temp\ipykernel_29044\3882918921.py:47: SettingWithCopyWarning: 
A value is trying to

In [57]:
# DATE PRE-PROCESSING

# working dataframes are:
# filt_elec_df, pumps_df, well_work_df, well_install_df, maintenance_df

# We could adjust the dates for well workovers, install, and maintenance to coincide with the month to make it easy
# Need to figure out the min and max dates for all df and adjust to the highest of the mins/lowest of the max

# Add ["Year-Month"] as a column to all dataframes for date-aligning 
# Use dt.to_period('M') to keep it time aware
filt_elec_df["Year-Month"] = pd.to_datetime(filt_elec_df["Usage Month"])
filt_elec_df["Year-Month"] = filt_elec_df["Year-Month"].dt.to_period("M")
pumps_df["Year-Month"] = pumps_df["Month End"].dt.to_period("M")
well_work_df["Year-Month"] = well_work_df["Date"].dt.to_period("M")
well_install_df["Year-Month"] = well_install_df["Install Date"].dt.to_period("M")
maintenance_df["Year-Month"] = pd.to_datetime(maintenance_df["Date Done"])
maintenance_df["Year-Month"] = maintenance_df["Year-Month"].dt.to_period("M")
#revenue_df["Year-Month"] = pd.to_datetime(revenue_df["Date"])
revenue_df["Year-Month"] = revenue_df["Date"].dt.to_period("M")


# Get date ranges
# Dataframes cannot be passed to a dictionary so we gotta do this the weird way
df_list = [filt_elec_df, pumps_df, well_work_df, well_install_df, maintenance_df, revenue_df]
name_list = ["elec_df", "pump_df", "well_work_df", "well_install_df", "maintenance_df", "revenue_df"]

for df, name in zip(df_list, name_list):
    print(f"{name}: Start date {df["Year-Month"].min()} | End date {df["Year-Month"].max()}")

# Trim dates
date_range = pd.date_range(start=filt_elec_df["Year-Month"].min().to_timestamp(), end=filt_elec_df["Year-Month"].max().to_timestamp(), freq="ME")
date_range = date_range.to_period("M")

# re-add the truncated last month
add_month = pd.PeriodIndex(["2025-09",], freq="M")
date_range = date_range.append(add_month)
print(date_range)

elec_df: Start date 2017-08 | End date 2025-09
pump_df: Start date 2002-03 | End date 2026-03
well_work_df: Start date 2017-01 | End date 2026-07
well_install_df: Start date 1991-07 | End date 2023-11
maintenance_df: Start date 1931-12 | End date 2026-03
revenue_df: Start date 2017-01 | End date 2026-05
PeriodIndex(['2017-08', '2017-09', '2017-10', '2017-11', '2017-12', '2018-01',
             '2018-02', '2018-03', '2018-04', '2018-05', '2018-06', '2018-07',
             '2018-08', '2018-09', '2018-10', '2018-11', '2018-12', '2019-01',
             '2019-02', '2019-03', '2019-04', '2019-05', '2019-06', '2019-07',
             '2019-08', '2019-09', '2019-10', '2019-11', '2019-12', '2020-01',
             '2020-02', '2020-03', '2020-04', '2020-05', '2020-06', '2020-07',
             '2020-08', '2020-09', '2020-10', '2020-11', '2020-12', '2021-01',
             '2021-02', '2021-03', '2021-04', '2021-05', '2021-06', '2021-07',
             '2021-08', '2021-09', '2021-10', '2021-11', '2021-

C:\Users\jroland\AppData\Local\Temp\ipykernel_29044\1803829176.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filt_elec_df["Year-Month"] = pd.to_datetime(filt_elec_df["Usage Month"])
C:\Users\jroland\AppData\Local\Temp\ipykernel_29044\1803829176.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filt_elec_df["Year-Month"] = filt_elec_df["Year-Month"].dt.to_period("M")


In [58]:
# DUPLICATE DATES PROCESSING
# Some dataframes may have multiple incidents recorded in the same month. 
# Need to explore handling this. 

# Get a count of incidents that have the same well name and date for suspecting dataframes
df_list = [filt_elec_df, pumps_df, well_work_df, well_install_df, maintenance_df]
name_list = ["elec_df", "pump_df", "well_work_df", "well_install_df", "maintenance_df"]

# maintenance_df is the only one that seems to now have multiple entries for a given month. Need to figure out
# appropriate means of handling. Columns of concern: Comments, Asset ID, and Total Cost. Comments and Asset ID 
# should be concatenated. Total Cost should be added.

# Combine records for maintenance_df
text_cols = ["Comments", "Asset ID#"]
money_col = ["Total Cost"]

# reset_index() to undo the multi-index assignment
grouped_maint_df = maintenance_df.groupby(["Name", "Year-Month"]).agg(
    total_cost=("Total Cost", "sum"),
    comments=("Comments", lambda x: " || ".join(x.dropna())),
    asset_id=("Asset ID#", lambda x: " || ".join(x.dropna()))).reset_index()

df_list = [filt_elec_df, pumps_df, well_work_df, well_install_df, grouped_maint_df]

print(grouped_maint_df)

for df, name in zip(df_list, name_list):
    # check that there are no duplicates in all dataframes
    counts = df.groupby(["Name", "Year-Month"]).size()
    print(counts[counts > 1])



           Name Year-Month  total_cost  \
0      ABNEY #2    2018-04     1166.00   
1      ABNEY #2    2018-07     3291.00   
2      ABNEY #2    2018-08     5493.80   
3      ABNEY #2    2018-12     5248.04   
4      ABNEY #2    2019-10     1859.45   
...         ...        ...         ...   
1571  WILSON #4    2022-03      682.45   
1572  WILSON #4    2022-06      563.28   
1573  WILSON #4    2022-07     1976.40   
1574  WILSON #4    2024-03      378.72   
1575  WILSON #4    2025-02      413.16   

                                               comments          asset_id  
0     8/15/2018 Mason Welch - Packing replacedPost: ...            P-687B  
1     7/27/2018 Mason Welch - 4.25" plungers replace...  P-687B || P-687B  
2     8/31/2018 Mason Welch - 4.25" packing replaced...            P-687B  
3     1/3/2019 Mason Welch - Rebuilt pump set at Abn...            P-167B  
4     10/8/2019 Mason Welch - Packing replaced and o...            P-167B  
...                                    

In [59]:
# MERGE PRE-PROCESSING - Electrical
                                                                              
print(filt_elec_df.shape)

# check number of entries by well
print(filt_elec_df.groupby("Name")["Year-Month"].size())
well_names = filt_elec_df["Name"].unique()

# append entries for Blum #6 and Blum #9 to allow for duplicating the billing data 
well_names = np.append(well_names, ["BLUM #6", "BLUM #9"])

elec_ind = pd.MultiIndex.from_product([well_names, date_range], names = ["Name", "Year-Month"])
print(elec_ind)

# set elec_df index to the same format as new multi-index
filt_elec_df = filt_elec_df.set_index(["Name", "Year-Month"])

# set multi-index to elec_df, inserting NA rows for entries where a month was missing
filt_elec_df = filt_elec_df.reindex(elec_ind)

# sort df if it wasn't already
filt_elec_df = filt_elec_df.sort_index(level=["Name", "Year-Month"], ascending=True)
print(filt_elec_df)

filt_elec_df = filt_elec_df.reset_index()


full_elec_df = filt_elec_df.fillna({"Final Bill": 0, "Metered DM" : 0,
                                    "Billed DM": 0, "Power Factor": 0,
                                    "Fuel Rate": 0})

# we need to interpolate the Billing KWH the same way as Hours - using the proportion of water total
# can't do that until we have joined to the pumps dataframe, though!
# 825 total NA records for Billing KWH
print(full_elec_df["Billing KWH"].isna().sum())

# make sure there are no more NA entries and check comparisons between wells
print(full_elec_df.isna().sum())

# NOTE: service address, account number, usage month, and bill month now have NA values - can probably just safely remove those columns now
# Remove the above columns
exclude = ["Service Address", "Account Number", "Usage Month", "Bill Month"]
full_elec_df = full_elec_df[[col for col in full_elec_df.columns if col not in exclude]]
print(full_elec_df.columns)



(9077, 12)
Name
1209 UNIT SHOP                   98
13 WINDING WAY UNIT (CLEMENS)    94
4-SIGHT MOORE                    97
ABNEY #2                         98
ABNEY #3/ABNEY PLANT             98
                                 ..
WILLIAMS #6                      98
WILLIAMS #7                      98
WILLIAMS #8                      98
WILLIAMS #9                      98
WILSON #4                        97
Name: Year-Month, Length: 99, dtype: int64
MultiIndex([('SHARP #9', '2017-08'),
            ('SHARP #9', '2017-09'),
            ('SHARP #9', '2017-10'),
            ('SHARP #9', '2017-11'),
            ('SHARP #9', '2017-12'),
            ('SHARP #9', '2018-01'),
            ('SHARP #9', '2018-02'),
            ('SHARP #9', '2018-03'),
            ('SHARP #9', '2018-04'),
            ('SHARP #9', '2018-05'),
            ...
            ( 'BLUM #9', '2024-12'),
            ( 'BLUM #9', '2025-01'),
            ( 'BLUM #9', '2025-02'),
            ( 'BLUM #9', '2025-03'),
           

In [60]:
# MERGE PRE-PROCESSING - Pumps
# pd.set_option("display.max_rows", None)
print("Total number of entries in pumps dataframe")
print(len(pumps_df))

# see which pumps have missing month entries
print("Count of entries per well")
print(pumps_df.groupby("Name")["Year-Month"].size())

# create full cartesian product of pump names and dates
pump_ind = pumps_df["Name"].unique()
cart_index = pd.MultiIndex.from_product([pump_ind, date_range],
                                        names=["Name", "Year-Month"])
print("Multi-index: ", cart_index)
# create multi-index from cartesian product
pumps_df = pumps_df.set_index(["Name", "Year-Month"])
pumps_df = pumps_df.reindex(cart_index)
pumps_df = pumps_df.sort_index(level=["Name", "Year-Month"], ascending=True)

print(pumps_df)
pumps_df = pumps_df.reset_index()
print("Total number of entries in pumps dataframe after reindexing")

# check reindex 
print(len(pumps_df))
print(pumps_df.groupby("Name")["Year-Month"].size())

# sort dataframe
pumps_df = pumps_df.sort_values(["Name", "Year-Month"])
print("Pumps dataframe ordering: ", pumps_df)

# inspect NA values
print("Number of entries with NA anywhere")
print(len(pumps_df[pumps_df.isna().any(axis=1)]))
print("Number of entries with NA in Hours")
print(len(pumps_df[pumps_df["Hours"].isna()]))
print("Number of entries with NA in Water Total")
print(len(pumps_df[pumps_df["Water Total"].isna()]))
print("Number of entries with NA in PSI")
print(len(pumps_df[pumps_df["PSI"].isna()]))

# NAs - Hours and PSI
# Entries which have NA for Hours or PSI where Water Total < 100 should be set to 0
print("Number of NA entries where Water Total < 100 for Hours or PSI")
print(len(pumps_df[(pumps_df["Hours"].isna() | pumps_df["PSI"].isna()) & (pumps_df["Water Total"] < 100)].sort_values(by="Year-Month", ascending=False)))
# Entries for which Hours or PSI where Water Total > 100 need to be inspected further
print("Number of NA entries where Water Total > 100 for Hours or PSI")
print(len(pumps_df[(pumps_df["Hours"].isna() | pumps_df["PSI"].isna()) & (pumps_df["Water Total"] > 100)]))
print("Number of NA entries where Water Total > 100 for Hours AND PSI")
print(len(pumps_df[(pumps_df["Hours"].isna() & pumps_df["PSI"].isna()) & (pumps_df["Water Total"] > 100)]))
print("Number of NA entries where Water Total < 100 for Hours")
print(len(pumps_df[(pumps_df["Hours"].isna()) & (pumps_df["Water Total"] < 100)]))
print("Number of NA entries where Water Total < 100 for PSI")
print(len(pumps_df[(pumps_df["PSI"].isna()) & (pumps_df["Water Total"] < 100)]))

Total number of entries in pumps dataframe
25911
Count of entries per well
Name
ABNEY #2        289
ABNEY #3        289
ALBERTSON #4    289
BASS #4         289
BASS #5         289
               ... 
WILLIAMS #6     289
WILLIAMS #7     289
WILLIAMS #8     289
WILLIAMS #9     289
WILSON #4       289
Name: Year-Month, Length: 94, dtype: int64
Multi-index:  MultiIndex([('ABNEY #2', '2017-08'),
            ('ABNEY #2', '2017-09'),
            ('ABNEY #2', '2017-10'),
            ('ABNEY #2', '2017-11'),
            ('ABNEY #2', '2017-12'),
            ('ABNEY #2', '2018-01'),
            ('ABNEY #2', '2018-02'),
            ('ABNEY #2', '2018-03'),
            ('ABNEY #2', '2018-04'),
            ('ABNEY #2', '2018-05'),
            ...
            (  'KEE #6', '2024-12'),
            (  'KEE #6', '2025-01'),
            (  'KEE #6', '2025-02'),
            (  'KEE #6', '2025-03'),
            (  'KEE #6', '2025-04'),
            (  'KEE #6', '2025-05'),
            (  'KEE #6', '2025-06')

In [61]:
# MERGE PRE-PROCESSING Pumps Continued
# 5/26/26 

# 6/4/26
# 547 entries where Water Total is NA, all of these have NA in PSI or Hours, but only 14 have NA for PSI AND Hours
# After inspection, there are 14 NA entries for PSI and 547 for Hours. 

# The psi reading is an average for the month, if I recall correctly. The only thing I can think of is to
# cross reference this against the billing data for the same time period, which requires joining the datasets.
# NOTE: I spoke to Mason, all of the following wells have been abandoned or plugged and can be dropped from 
# analysis: Cole 5, Colville 2, Fields 5, Sharp 5, Williams 3, Williams 4, and Williams 5.
water_filter = pumps_df["Water Total"].isna()
print("NA entries for Water Total")
print(len(pumps_df[water_filter]))

# Check whether the entries with NA for water total also have NA for Hours or psi
water_filter = pumps_df["Water Total"].isna() & (pumps_df["PSI"].isna() | pumps_df["Hours"].isna())
print("NA entries for Water Total where PSI OR Hours have NA entries")
print(len(pumps_df[water_filter]))

# Check whether the entries with NA for water total also have NA for Hours AND psi
water_filter = pumps_df["Water Total"].isna() & (pumps_df["PSI"].isna() & pumps_df["Hours"].isna())
print("NA entries for Water Total where PSI AND Hours have NA entries")
print(len(pumps_df[water_filter]))

# 14 entries where PSI is NA
water_filter = pumps_df["Water Total"].isna() & (pumps_df["PSI"].isna())
print("NA entries for Water Total where PSI is NA")
print(len(pumps_df[water_filter]))

# 547 where Hours is NA
print("NA entries for Water Total where Hours is NA")
water_filter = pumps_df["Water Total"].isna() & pumps_df["Hours"].isna()
print(len(pumps_df[water_filter]))

na_water_df = pumps_df[water_filter]
# export to excel sheet for easier viewing
print(na_water_df.to_excel("Missing Water Totals.xlsx"))

# Drop abandoned or plugged wells.
water_filter = pumps_df["Water Total"].isna() & (pumps_df["PSI"].isna() | pumps_df["Hours"].isna())
drop_wells = pumps_df.loc[water_filter, "Name"]
print(drop_wells)

# only include entries from pumps df that are not in drop wells
pumps_df = pumps_df[~pumps_df["Name"].isin(drop_wells)]
water_filter = pumps_df["Water Total"].isna()
print("NA entries for Water Total after dropping abandoned or plugged wells.")
print(len(pumps_df[water_filter]))


# At present, 172 entries for PSI with Water Total < 100 have NA, they can probably be inferred as 0
psi_filter = pumps_df["PSI"].isna() & (pumps_df["Water Total"] < 100)
pumps_df.loc[psi_filter, "PSI"] = 0
print("NA entries for PSI where Water Total < 100")
print(len(pumps_df[psi_filter]))

# 183 entries where Water Total < 100 have NA for Hours, those can also probably be inferred as 0
hours_filter = pumps_df["Hours"].isna() & (pumps_df["Water Total"] < 100)
pumps_df.loc[hours_filter, "Hours"] = 0
print("NA entries for Hours where Water Total < 100")
print(len(pumps_df[hours_filter]))

# There are 5 entries for PSI with NA where Water Total is > 100, these probably need to be interpolated.
# Interpolation strategy - 3 month rolling average, where at least 2/3 values need to be included.
# Implementation strategy - 1) reindex pumps_df to use Year-Month as the index 
#                         - 2) create column "Avg_PSI_6Mo"
#                         - 3) average based on the previous and following index and set the entry to the result
#                         - 4) Set PSI to Avg_PSI_6Mo where PSI is NA
    
psi_filter = pumps_df["PSI"].isna() & (pumps_df["Water Total"] > 100)
print(pumps_df[psi_filter])

print(pumps_df)

def converge_psi(df):
    "Calculates rolling averages on PSI until there are no NA entries left for psi"
    # rolling() returns a rolling() object
    # create column "Avg_PSI_6Mo" and use a rolling average
    # We have to group on the well name to ensure it's not calculating across multiple wells
    while(df["PSI"].isna().any()):  
        psi_filter = df["PSI"].isna()
        df["Avg_PSI_6Mo"] = (
            df
            .groupby("Name")["PSI"]
            .rolling(window=6, min_periods=1)
            .mean()
            .reset_index(drop=True)
        )
        # Set PSI to Avg_PSI_6Mo where PSI is NA
        df.loc[psi_filter, "PSI"] = df.loc[psi_filter, "Avg_PSI_6Mo"] 
        # re-filter after adjustment
        psi_filter = df["PSI"].isna()
        
        df["Avg_PSI_12Mo"] = (
            df
            .groupby("Name")["PSI"]
            .rolling(window=12, min_periods=2)
            .mean()
            .reset_index(drop=True)
        )
        # Set PSI to Avg_PSI_12Mo where PSI is NA
        df.loc[psi_filter, "PSI"] = df.loc[psi_filter, "Avg_PSI_12Mo"]
        psi_filter = df["PSI"].isna()
    return df

pumps_df = converge_psi(pumps_df)

# There are 1167 entries for Hours with NA where Water Total is > 100, these also probably should be interpolated.
# Interpolation strategy - We want to estimate the run hours of a missing entry for the current month. We have water volume as the only available data 
#                          that directly relates to run hours. In order to estimate run hours from water volume, we need to calculate the hours/volume
#                          ratio for each month, then apply a 6 month rolling average. We're using a 6 month to account for the possibility that there 
#                          may be multiple consecutive entries where Hours is NA.

# Interpolation implementation - 1) Create a column "Hours/Water Ratio" ratio of Hours to Water Total. This will be hours/bbl. 
#                        - 2) Create another column "6Mo Avg Hours/Water" as a 6 month rolling average, where at least 3/6 need to be included. 
#                        - 3) Calculate Hours by multiplying Water Total by the rolling average for the months where Hours is NA.

def converge_hours(df):
    "Calculates rolling averages on Hours until there are no NA entries left for Hours"
    # sentinal value for preventing infinite loops
    previous_na = 0
    while(df["Hours"].isna().any()):
        current_na = df["Hours"].isna().sum()

        # converged but test condition didn't execute correctly
        if current_na == 0:
            break

        # converged by no improvement
        if current_na == previous_na:
            print("converge_hours converged without reaching 0 NA values")
            break

        previous_na = current_na
            
        hours_filter = df["Hours"].isna()
        df.loc[~(df["Water Total"]==0), "Hours/Water Ratio"] = (
            df["Hours"] / df["Water Total"]
        )
           
        df["6Mo Avg Hours/Water"] = (
            df["Hours/Water Ratio"]
            .rolling(window=6, min_periods=1)
            .mean()
        )
    
        df["12Mo Avg Hours/Water"] = (
            df["Hours/Water Ratio"]
            .rolling(window=12, min_periods=1)
            .mean()
        )
    
        df["24Mo Avg Hours/Water"] = (
            df["Hours/Water Ratio"]
            .rolling(window=24, min_periods=1)
            .mean()
        )
    
        df.loc[hours_filter, "Hours"] = (
            df.loc[hours_filter, "6Mo Avg Hours/Water"]
            * df.loc[hours_filter, "Water Total"]
        )
        hours_filter = df["Hours"].isna()
        print("After 6Mo average hours/water", len(df[hours_filter]))
    
        df.loc[hours_filter, "Hours"] = (
            df.loc[hours_filter, "12Mo Avg Hours/Water"]
            * df.loc[hours_filter, "Water Total"]
        )
        hours_filter = df["Hours"].isna()
        print("After 12Mo average hours/water", len(df[hours_filter]))
        
        df.loc[hours_filter, "Hours"] = (
            df.loc[hours_filter, "24Mo Avg Hours/Water"]
            * df.loc[hours_filter, "Water Total"]
        )
        hours_filter = df["Hours"].isna()
        print("After 24Mo average hours/water", len(df[hours_filter]))
        
        abney_filt = (df["Name"] == "ABNEY #2") & (df["Hours"].isna())
        if abney_filt.any():
            df.loc[abney_filt, "Hours"] = (
                df["Hours/Water Ratio"].mean()
                * df.loc[abney_filt, "Water Total"]
            )
            
    return df

pumps_df = converge_hours(pumps_df)

hours_filter = pumps_df["Hours"].isna()

# if hours converged and there are still na values left, fill them with the average
pumps_df.loc[hours_filter, "Hours"] = (
    pumps_df["Hours/Water Ratio"].mean()
    * pumps_df.loc[hours_filter, "Water Total"]
)

psi_filter = pumps_df["PSI"].isna()
hours_filter = pumps_df["Hours"].isna()
print("Number of NA entries for Hours after processing ")
print(len(pumps_df[hours_filter]))
print("Number of NA entries for PSI after processing ")
print(len(pumps_df[psi_filter]))
print("Number of NA entries for Water total after processing ")
print(len(pumps_df[water_filter]))
print("Number of NA entries anywhere after processing.")
print(len(pumps_df[pumps_df.isna().any(axis=1)]))

# 6/8/2026
# Re-running script once produces no NA entries for Hours or PSI remaining.
print("Final count of all entries after processing.")
print(len(pumps_df))

NA entries for Water Total
547
NA entries for Water Total where PSI OR Hours have NA entries
547
NA entries for Water Total where PSI AND Hours have NA entries
547
NA entries for Water Total where PSI is NA
547
NA entries for Water Total where Hours is NA
547
None
2005        COLE #5
2006        COLE #5
2007        COLE #5
2008        COLE #5
2009        COLE #5
           ...     
8717    WILLIAMS #5
8718    WILLIAMS #5
8719    WILLIAMS #5
8720    WILLIAMS #5
8721    WILLIAMS #5
Name: Name, Length: 547, dtype: object
NA entries for Water Total after dropping abandoned or plugged wells.
0
NA entries for PSI where Water Total < 100
146
NA entries for Hours where Water Total < 100
155
             Name Year-Month  Month End         Well  Hours  Water Total  PSI
590    BERWALD #5    2017-10 2017-10-31   BERWALD #5    NaN      38354.0  NaN
593    BERWALD #5    2018-01 2018-01-31   BERWALD #5    NaN      16312.0  NaN
1373  BOLDRICK #1    2017-09 2017-09-30  BOLDRICK #1    NaN      11199.0  

In [62]:
# MERGE PRE-PROCESSING Pumps Continued
# 6/11/2026
# Remove columns no longer needed
exclude = ['Month End', 'Avg_PSI_6Mo', 'Avg_PSI_12Mo', 'Hours/Water Ratio', '6Mo Avg Hours/Water', '12Mo Avg Hours/Water',
           '24Mo Avg Hours/Water', '36Mo Avg Hours/Water', '48Mo Avg Hours/Water']
include = [col for col in pumps_df.columns if col not in exclude]
pumps_df = pumps_df[include]
pumps_df

,Name,Year-Month,Well,Hours,Water Total,PSI
0,ABNEY #2,2017-08,ABNEY #2,883.922017,503920.0,772.857143
1,ABNEY #2,2017-09,ABNEY #2,910.129905,518861.0,777.142857
2,ABNEY #2,2017-10,ABNEY #2,926.036011,527929.0,782.857143
3,ABNEY #2,2017-11,ABNEY #2,884.029017,503981.0,774.285714
4,ABNEY #2,2017-12,ABNEY #2,834.928475,475989.0,773.333333
...,...,...,...,...,...,...
9207,WILSON #4,2025-05,WILSON #4,748.000000,218425.0,754.440000
9208,WILSON #4,2025-06,WILSON #4,713.820000,213333.0,772.220000
9209,WILSON #4,2025-07,WILSON #4,139.210000,41763.0,810.000000
9210,WILSON #4,2025-08,WILSON #4,0.000000,0.0,0.000000


In [63]:
# MERGE PRE-PROCESSING pump and revenue data

# We have monthly records of revenue
# We need to assign revenue to each well according to the proportion of the total water disposed
# Strategy - assign the full dollar amount to each well for each month, duplicating the data.
# Then create a new column representing the percent of the monthly water total for that well, multiply the full dollar value by that proportion

# transform(func) returns a dataframe of the same shape as the original. It takes the aggregated values and the re-distributes
# the result of the function applied to the aggregated data back to the individual rows.
pumps_df["Monthly Water Percentage"] = ((pumps_df["Water Total"])
                                       /(pumps_df.groupby("Year-Month")["Water Total"].transform("sum")))

# assigns total monthly revenue value to every well
pumps_df = pumps_df.merge(revenue_df, on="Year-Month", how="left")

# distributes the total monthly revenue value according to the proportion of water disposed by that well
pumps_df["Revenue"] = pumps_df["Revenue"]*pumps_df["Monthly Water Percentage"]
print(pumps_df.groupby("Year-Month")["Revenue"].sum())


Year-Month
2017-08     939709.03
2017-09     960565.78
2017-10     985174.60
2017-11    1009611.79
2017-12    1060238.94
              ...    
2025-05    1786444.81
2025-06    1720279.26
2025-07    1778005.29
2025-08    1754916.96
2025-09    1725763.51
Freq: M, Name: Revenue, Length: 98, dtype: float64


In [64]:
# MERGE PRE-PROCESSING well work 
# No NA values present
print(well_work_df.columns)
print(len(well_work_df[well_work_df.isna().any(axis=1)]))

# Create missing entries to get dataframe to be the same size as pumps and billing dfs. 
# That count is 8,526. Electrical dataframe has a higher count because it is including sites that
# we are not including in analysis and will be filtered later.

# create full cartesian product of pump names and dates
pump_ind = well_work_df["Name"].unique()
cart_index = pd.MultiIndex.from_product([pump_ind, date_range],
                                        names=["Name", "Year-Month"])
print("Multi-index: ", cart_index)
# create multi-index from cartesian product
well_work_df = well_work_df.set_index(["Name", "Year-Month"])
well_work_df = well_work_df.reindex(cart_index)
well_work_df = well_work_df.sort_index(level=["Name", "Year-Month"], ascending=True)

print(well_work_df)
well_work_df = well_work_df.reset_index()
print("Total number of entries in well work dataframe after reindexing")

# check reindex 
print(len(well_work_df))
print(well_work_df.groupby("Name")["Year-Month"].size())

Index(['Date', 'Well', 'Info', 'Cost', 'Name', 'Year-Month'], dtype='object')
0
Multi-index:  MultiIndex([( 'ABNEY #3', '2017-08'),
            ( 'ABNEY #3', '2017-09'),
            ( 'ABNEY #3', '2017-10'),
            ( 'ABNEY #3', '2017-11'),
            ( 'ABNEY #3', '2017-12'),
            ( 'ABNEY #3', '2018-01'),
            ( 'ABNEY #3', '2018-02'),
            ( 'ABNEY #3', '2018-03'),
            ( 'ABNEY #3', '2018-04'),
            ( 'ABNEY #3', '2018-05'),
            ...
            ('WILSON #4', '2024-12'),
            ('WILSON #4', '2025-01'),
            ('WILSON #4', '2025-02'),
            ('WILSON #4', '2025-03'),
            ('WILSON #4', '2025-04'),
            ('WILSON #4', '2025-05'),
            ('WILSON #4', '2025-06'),
            ('WILSON #4', '2025-07'),
            ('WILSON #4', '2025-08'),
            ('WILSON #4', '2025-09')],
           names=['Name', 'Year-Month'], length=4802)
                     Date Well Info  Cost
Name      Year-Month             

In [65]:
# MERGE PRE-PROCESSING well installs
print(len(well_install_df))
print(well_install_df.columns)
print(len(well_install_df[well_install_df.isna().any(axis=1)]))
print(well_install_df[well_install_df.isna().any(axis=1)])

# Notes
# NaNs are probably not an issue here, but will need to remember that they exist in the analysis phase.
# In this dataset, it makes sense that there might not be records for a given pump.

# create full cartesian product of pump names and dates
pump_ind = well_install_df["Name"].unique()
cart_index = pd.MultiIndex.from_product([pump_ind, date_range],
                                        names=["Name", "Year-Month"])
print("Multi-index: ", cart_index)
# create multi-index from cartesian product
well_install_df = well_install_df.set_index(["Name", "Year-Month"])
well_install_df = well_install_df.reindex(cart_index)
well_install_df = well_install_df.sort_index(level=["Name", "Year-Month"], ascending=True)

print(well_install_df)
well_install_df = well_install_df.reset_index()
print("Total number of entries in well install dataframe after reindexing")

# check reindex 
print(len(well_install_df))
print(well_install_df.groupby("Name")["Year-Month"].size())

90
Index(['Well Name', 'Pump Model', 'Pump Number', 'Plunger Size',
       'Install Date', 'Name', 'Year-Month'],
      dtype='object')
21
       Well Name      Pump Model Pump Number  Plunger Size Install Date  \
0       ABNEY #2             NaN         NaN           NaN          NaT   
1       ABNEY #3  M520A C-CT-ARZ      P-660B           NaN   2019-05-25   
9   BLACKWELL #3  M520A C-CT-ARZ      P-680B           NaN   2014-12-03   
11       BLUM #6  M520A C-CT-ARZ      P-692B           NaN   2016-01-04   
13       BLUM #9  M520A C-CT-ARZ      P-683B           NaN   2019-07-16   
18      BOLES #1  M520A C-CT-ARZ      P-670B           NaN   2014-04-08   
20       COLE #5       Abandoned         NaN           NaN          NaT   
30     CUNYUS #6  M520A C-CT-ARZ      P-710B           NaN   2023-09-22   
34    DOWNING #4  M520A C-CT-ARZ      P-682B           NaN   2021-02-25   
35    DOWNING #5  M520A C-CT-ARZ         NaN           NaN          NaT   
42      HAMON #4  M520A C-CT-ARZ    

In [66]:
# MERGE PRE-PROCESSING well maintenance
print(len(grouped_maint_df))
print(len(grouped_maint_df[grouped_maint_df.isna().any(axis=1)]))
print(grouped_maint_df)

# No NaNs here, good job everybody.

# Need to get datasets to all be the same size for joining

# create full cartesian product of pump names and dates
pump_ind = grouped_maint_df["Name"].unique()
cart_index = pd.MultiIndex.from_product([pump_ind, date_range],
                                        names=["Name", "Year-Month"])
print("Multi-index: ", cart_index)
# create multi-index from cartesian product
grouped_maint_df = grouped_maint_df.set_index(["Name", "Year-Month"])
grouped_maint_df = grouped_maint_df.reindex(cart_index)
grouped_maint_df = grouped_maint_df.sort_index(level=["Name", "Year-Month"], ascending=True)

print(grouped_maint_df)
grouped_maint_df = grouped_maint_df.reset_index()
print("Total number of entries in maintenance dataframe after reindexing")

# check reindex 
print(len(grouped_maint_df))
print(grouped_maint_df.groupby("Name")["Year-Month"].size())

1576
0
           Name Year-Month  total_cost  \
0      ABNEY #2    2018-04     1166.00   
1      ABNEY #2    2018-07     3291.00   
2      ABNEY #2    2018-08     5493.80   
3      ABNEY #2    2018-12     5248.04   
4      ABNEY #2    2019-10     1859.45   
...         ...        ...         ...   
1571  WILSON #4    2022-03      682.45   
1572  WILSON #4    2022-06      563.28   
1573  WILSON #4    2022-07     1976.40   
1574  WILSON #4    2024-03      378.72   
1575  WILSON #4    2025-02      413.16   

                                               comments          asset_id  
0     8/15/2018 Mason Welch - Packing replacedPost: ...            P-687B  
1     7/27/2018 Mason Welch - 4.25" plungers replace...  P-687B || P-687B  
2     8/31/2018 Mason Welch - 4.25" packing replaced...            P-687B  
3     1/3/2019 Mason Welch - Rebuilt pump set at Abn...            P-167B  
4     10/8/2019 Mason Welch - Packing replaced and o...            P-167B  
...                             

In [67]:
# MERGE PRE-PROCESSING 
# Data Joining
# Check that all data has been pruned to the same date range after re-indexing
# Check whether all dataframes are the same length
df_list = [full_elec_df, pumps_df, well_work_df, well_install_df, grouped_maint_df]
name_list = ["elec_df", "pump_df", "well_work_df", "well_install_df", "maintenance_df"]

for df, name in zip(df_list, name_list):
    print(f"{name}: Start date {df["Year-Month"].min()} | End date {df["Year-Month"].max()}")
    print(name, " number entries:", len(df))

# We want to join on Name and Year-Month

print("Well names before pruning and adjusting")
for df, name in zip(df_list, name_list):
    print(name, " well names list:", df["Name"].unique())
    df.loc[:, "Name"] = df["Name"].str.upper()
    
# 6/18/26 At present mostly the electrical df needs the most pruning
# NOTE - for the wells which have electrical billing co-mingled with a pit or other well
# that co-mingled status needs to be indicated in the final analysis.
full_elec_df["Combined_Bill?"] = "No"
comingled_wells = ("BLUM PLANT/BLUM #9/BLUM #6", "BLACKWELL CENTER/BLACKWELL #3", "THOMPSON #1/THOMPSON PLANT")
#full_elec_df[full_elec_df["Name"].isin(comingled_wells), "Combined_Bill?"] = "Yes"
comingle_filt = full_elec_df["Name"].isin(comingled_wells)
full_elec_df.loc[comingle_filt, "Combined_Bill?"] = "Yes"

# adjust names for comingled entries to better match pumps_df for joining
full_elec_df.loc[full_elec_df["Name"] == "BLACKWELL CENTER/BLACKWELL #3", "Name"] = "BLACKWELL #3"
full_elec_df.loc[full_elec_df["Name"] == "THOMPSON #1/THOMPSON PLANT", "Name"] = "THOMPSON #1"

# NOTE - for the Blum #6 and Blum #9 we could do the following. 
# 1) Create a new entry in the pumps, well work, well install, and well maintenance dataframes that adds together all the values for both wells.
# It will not supersede the entries for the two wells separately, but will allow for analysis from
# both the perspective of duplicating the billing and analyzing against the individual well characteristics
# as well as combining the well characteristics to analyze against the correct billing data.
# 2) Create duplicate entries for the two wells separate from the comingled entry in the electrical data.
# NOTE - not doing this presently but leaving this idea here for later.

# create duplicate entries for Blum 6 and 9 in the electrical billing data
blum6_filt = full_elec_df["Name"] == "BLUM #6"
blum9_filt = full_elec_df["Name"] == "BLUM #9"
blum_com_filt = full_elec_df["Name"] == "BLUM PLANT/BLUM #9/BLUM #6"

copy_fields = [col for col in full_elec_df.columns if col != "Name"]
full_elec_df.loc[blum6_filt, copy_fields] = full_elec_df.loc[blum_com_filt, copy_fields].values
full_elec_df.loc[blum9_filt, copy_fields] = full_elec_df.loc[blum_com_filt, copy_fields].values

# check that data duplication was successful
print(full_elec_df[blum6_filt])
    
# replace Pew Unit6LGV with Pew #6
pew_filt = full_elec_df["Name"] == "PEW UNIT6LGV"
full_elec_df.loc[pew_filt, "Name"] = "PEW #6"

# replace Pew Plant/Pew #5 with just Pew #5
full_elec_df.loc[full_elec_df["Name"] == "PEW PLANT/PEW #5", "Name"] = "PEW #5"

# Replace Abney #3/Abney Plant with just Abney #3
full_elec_df.loc[full_elec_df["Name"] == "ABNEY #3/ABNEY PLANT", "Name"] = "ABNEY #3"

# Replace Arco Cole/Colville #3 with just Colville #3
full_elec_df.loc[full_elec_df["Name"] == "ARCO COLE CENTER/COLVILLE #3", "Name"] = "COLVILLE #3"

# Replace Hughes #3/Hughes Plant with just Hughes #3
full_elec_df.loc[full_elec_df["Name"] == "HUGHES #3/HUGHES PLANT", "Name"] = "HUGHES #3"

# There are others that could be removed at this stage, but they will get filtered out
# when joining based on valid_wells
invalid = ["HUGHES PLANT", "LAKE PLANT/LAKE #1", "LAKE #1", "WILLIAMS #1", "WILLIAMS #2"]

print("Well names after pruning and adjusting.")
for df, name in zip(df_list, name_list):
#    remove invalid wells
    df = df[~df["Name"].isin(invalid)]
    print(name, " well names list:", df["Name"].unique())

    
# Note issues and inconsistencies here
# 1) All text needs to be set to same case and format
# 2) A) Electrical dataframe has the following mingled
# -  Abney #3/Abney Plant
# - Arco Cole Center/Colville #3
# - Blackwell Center/Blackwell #3
# - Blum Plant/Blum #9/Blum #6
# - Hughes #3/Hughes Plant
# - Lake Plant/Lake #1
# - Pew Plant/Pew #5"
# - Thomson #1/Thompson Plant
# - Williams #1 & #2
    
#    B) This affects the following individual wells
# - Abney #3
# - Colville #3
# - Blackwell #3
# - Blum #6
# - Blum #9
# - Hughes #3
# - Lake #1
# - Pew #5
# - Thompson #1
# - Williams #1 
# - Williams #2
    
#    C) Ideas and potential solutions
# - Drop these wells from analysis and consideration - probably the easiest solution
# - Find the proportion of kwh usage that the individual wells and pits are responsible for - probably not feasible.
# - Duplicate these records for the individual wells - Blum #9 and Blum #6 would both get the billing data for Blum Plant/9/6

# Notes from Mason
# Hughes Plant does not exist anymore
# Abney Plant does not exist anymore
# Blum will probably be duplicated among sites
# Arco Cole should be minimal part of the Colville #3 cost
# Blackwell will reflect Blackwell Center alongside it
# Lake #1 and Lake Plant can be dropped entirely 
# Pew plant should be minimal part of the Pew #5 
# Pew Unit6LGV should be renamed to Pew #6 
# Thompson will reflect Thompson Center alongside it 
# Williams #1 has been down for most of this time period and can be disregarded. It was brought back up around March 2026.
# Sunkel Plant/Sunkel #3 are comingled
# Kee Plant/Kee #6 are comingled

elec_df: Start date 2017-08 | End date 2025-09
elec_df  number entries: 9898
pump_df: Start date 2017-08 | End date 2025-09
pump_df  number entries: 8526
well_work_df: Start date 2017-08 | End date 2025-09
well_work_df  number entries: 4802
well_install_df: Start date 2017-08 | End date 2025-09
well_install_df  number entries: 8820
maintenance_df: Start date 2017-08 | End date 2025-09
maintenance_df  number entries: 8918
Well names before pruning and adjusting
elec_df  well names list: ['1209 UNIT SHOP' '13 WINDING WAY UNIT (CLEMENS)' '4-SIGHT MOORE'
 'ABNEY #2' 'ABNEY #3/ABNEY PLANT' 'ALBERTSON #4' 'ANDREWS2 CENTER'
 'ARCO COLE CENTER/COLVILLE #3' 'BASS #4' 'BASS #5' 'BERWALD #4'
 'BERWALD #5' 'BERWALD #6' 'BLACKWELL #2' 'BLACKWELL CENTER/BLACKWELL #3'
 'BLUM #5' 'BLUM #6' 'BLUM #7' 'BLUM #9' 'BLUM PLANT/BLUM #9/BLUM #6'
 'BOLDRICK #1' 'BOLDRICK #2' 'BOLDRICK #3' 'BOLDRICK #4' 'BOLES #1'
 'BRADFORD1 CENTER' 'BRADFORD3 CENTER' 'CHRISTIAN #1' 'CITY TANK1'
 'CITY TANK3' 'COLE #5' 'COLVIL

In [68]:
# MERGE PRE-PROCESSING
# Data Joining Continued

# 6/24/26
# Should only need to join the records together based on well names in valid wells and year-month.
# All dataframes should be ordered from earliest to latest entries, grouped by well name and year-month.
# Final dataframe should have 87 wells * 98 entries per well = 8,526 entries in the joined table.

# Instead of trying to perform SQL-style joins, we can just concatenate the tables together horizontally.
# All dataframes need to have the same multi index for this to work properly.

valid_wells = pumps_df["Name"].unique()

# 1) Test to see if there wells in valid_wells not in each dataframe
#    or wells in the dataframe not in valid_wells
for df, name in zip(df_list, name_list):
   # df = df[df["Name"].isin(valid_wells)]
    print(name, " ", df)
    print(len(df["Name"].unique()))
    # this doesn't actually change the contents of df in df_list
    # df[filter] creates a new object, df is assigned to that new object
    # but it does not affect df_list as assignment in python
    # evaluates the right side and then binds the left side to the address of the right
    # while the elements in df_list still point to the old objects
    df = df[df["Name"].isin(valid_wells)]
    print(len(df["Name"].unique()))

# Need to verify that naming is consistent across datasets 
# and make sure that wells which should be included aren't being dropped. 

# need to get the difference between well names sets among the dataframes
pumps_set = set(valid_wells)
elec_set = set(full_elec_df["Name"].unique())
work_set = set(well_work_df["Name"].unique())
install_set = set(well_install_df["Name"].unique())
maint_set = set(grouped_maint_df["Name"].unique())

# list to compare to pump set
set_list = [elec_set, work_set, install_set, maint_set]
name_list = ["electrical", "well work", "well install", "well maintenance"]
for item, name in zip(set_list, name_list):
    print("Wells in pumps list not in ", name)
    print(sorted(pumps_set - item))
    print()
    print("Wells in ", name, " not in pumps list")
    print(sorted(item - pumps_set))
    print()

# 2) Filter out non-valid wells for each dataframe
# we have to use enumerate and array indexing to overwrite 
# values within a list. df = df[filter] inside of a loop
# creates a new address for df and assigns df[filter] to it, 
# rather than overwriting the original value at the address for df.
for i, (df, name) in enumerate(zip(df_list, name_list)):
    df_list[i] = df[df["Name"].isin(valid_wells)]
    print(name, len(df["Name"].unique()))

# 7/7/2026 final filtered sets are pumps - 87, electrical - 82,
# well work - 87, well install - 87, well maintenance - 87

elec_df                   Name Year-Month  Metered DM  Power Factor  Billed DM  \
0     1209 UNIT SHOP    2017-08        20.0         100.0       20.0   
1     1209 UNIT SHOP    2017-09        21.0         100.0       21.0   
2     1209 UNIT SHOP    2017-10        18.0         100.0       18.0   
3     1209 UNIT SHOP    2017-11        18.0         100.0       18.0   
4     1209 UNIT SHOP    2017-12        18.0         100.0       18.0   
...              ...        ...         ...           ...        ...   
9893       WILSON #4    2025-05        81.9          99.7       81.9   
9894       WILSON #4    2025-06        82.3          99.3       82.3   
9895       WILSON #4    2025-07        85.7          98.9       85.7   
9896       WILSON #4    2025-08         0.1          44.7        0.1   
9897       WILSON #4    2025-09         0.0          31.7        0.1   

      Billing KWH  Fuel Rate  Final Bill Combined_Bill?  
0          5232.0   0.028841      327.57             No  
1        

In [69]:
# MERGE PRE-PROCESSING
# Data Joining Continued
df_list = [full_elec_df, pumps_df, well_work_df, well_install_df, grouped_maint_df]

# join based on Name and Year-Month
merge_df = pumps_df.merge(full_elec_df, how="left", on=["Name", "Year-Month"])
print("After joining pumps and electrical data: ", merge_df)
print("Data length: ", len(merge_df))
print()

merge_df = merge_df.merge(well_work_df, how="left", on=["Name", "Year-Month"])
print("After joining pumps, electrical, and well work: ", merge_df)
print("Data length: ", len(merge_df))
print()

merge_df = merge_df.merge(well_install_df, how="left", on=["Name", "Year-Month"])
print("After joining pumps, electrical, well work, and well install: ", merge_df)
print("Data length: ", len(merge_df))
print()

merge_df = merge_df.merge(grouped_maint_df, how="left", on=["Name", "Year-Month"])
print("After all data join: ", merge_df)
print("Data length: ", len(merge_df))
print()

After joining pumps and electrical data:             Name Year-Month       Well       Hours  Water Total         PSI  \
0      ABNEY #2    2017-08   ABNEY #2  883.922017     503920.0  772.857143   
1      ABNEY #2    2017-09   ABNEY #2  910.129905     518861.0  777.142857   
2      ABNEY #2    2017-10   ABNEY #2  926.036011     527929.0  782.857143   
3      ABNEY #2    2017-11   ABNEY #2  884.029017     503981.0  774.285714   
4      ABNEY #2    2017-12   ABNEY #2  834.928475     475989.0  773.333333   
...         ...        ...        ...         ...          ...         ...   
8521  WILSON #4    2025-05  WILSON #4  748.000000     218425.0  754.440000   
8522  WILSON #4    2025-06  WILSON #4  713.820000     213333.0  772.220000   
8523  WILSON #4    2025-07  WILSON #4  139.210000      41763.0  810.000000   
8524  WILSON #4    2025-08  WILSON #4    0.000000          0.0    0.000000   
8525  WILSON #4    2025-09  WILSON #4    0.000000          0.0    0.000000   

      Monthly Water P

In [70]:
# cols to include
merge_df["Electrical Bill"] = merge_df["Final Bill"]
merge_df["Workover Cost"] = merge_df["Cost"]
merge_df["Maintenance Cost"] = merge_df["total_cost"]
merge_df["Workover Info"] = merge_df["Info"]
merge_df["Maintenance Comments"] = merge_df["comments"]

# interpolate Billing KWH with Water Total
# proportion of total Billing KWH should be close to proportion of total Water Volume
sum_kwh = merge_df["Billing KWH"].sum()
merge_df["Billing KWH"] = np.where(
    merge_df["Billing KWH"].isna(),
    sum_kwh * merge_df["Monthly Water Percentage"],
    merge_df["Billing KWH"]
)
print(merge_df["Billing KWH"].isna().sum())

include = ["Name", "Year-Month", "Hours", "Water Total", "PSI", "Revenue", "Metered DM",  
 "Power Factor", "Billed DM", "Billing KWH", "Fuel Rate", "Electrical Bill", "Workover Info", "Workover Cost", "Pump Model",
 "Pump Number", "Plunger Size", "Install Date", "Maintenance Cost", "Maintenance Comments"]

merge_df = merge_df[include]

# Fill NAs for costs with 0
merge_df["Electrical Bill"] = np.where(
    merge_df["Electrical Bill"].isna(),
    0,
    merge_df["Electrical Bill"]
)

merge_df["Maintenance Cost"] = np.where(
    merge_df["Maintenance Cost"].isna(),
    0,
    merge_df["Maintenance Cost"]
)

merge_df["Workover Cost"] = np.where(
    merge_df["Workover Cost"].isna(),
    0,
    merge_df["Workover Cost"]
)



0


In [71]:
merge_df.to_excel("All Data.xlsx")